[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mongodb-developer/GenAI-Showcase/blob/main/notebooks/agents/agent_fireworks_ai_langchain_mongodb.ipynb)

[![查看文章](https://img.shields.io/badge/View%20Article-blue)](https://www.mongodb.com/developer/products/atlas/agent-fireworksai-mongodb-langchain/)

## 安装库

In [ ]:
!pip install langchain langchain_openai langchain-fireworks langchain-mongodb arxiv pymupdf datasets pymongo

## 设置环境变量

You can set environment variables in a few ways.

您可以通过几种方式设置环境变量。

### Using the `env` command

### 使用 `env` 命令

The `env` command allows you to run a command with a modified environment.

`env` 命令允许您在修改后的环境中运行命令。

```bash
env VARIABLE_NAME=value command
```

For example, to set the `NODE_ENV` environment variable to `production` for a single command:

例如，为单个命令将 `NODE_ENV` 环境变量设置为 `production`：

```bash
env NODE_ENV=production node app.js
```

### Using `.env` files

### 使用 `.env` 文件

You can create a `.env` file in the root of your project to store environment variables. This is a common practice for managing configuration.

您可以在项目根目录创建一个 `.env` 文件来存储环境变量。这是管理配置的常见做法。

Create a file named `.env` in the root of your project:

在项目根目录创建一个名为 `.env` 的文件：

```
VARIABLE_NAME=value
ANOTHER_VARIABLE=another_value
```

Then, in your application code, you can load these variables using a library like `dotenv`.

然后，在您的应用程序代码中，您可以使用像 `dotenv` 这样的库加载这些变量。

First, install `dotenv`:

首先，安装 `dotenv`：

```bash
npm install dotenv
# or
yarn add dotenv
```

Then, require and configure `dotenv` at the top of your application's entry point file (e.g., `app.js` or `server.js`):

然后，在应用程序的入口文件（例如 `app.js` 或 `server.js`）的顶部引入并配置 `dotenv`：

```javascript
require('dotenv').config();

// Now you can access your environment variables
console.log(process.env.VARIABLE_NAME); // Output: value
console.log(process.env.ANOTHER_VARIABLE); // Output: another_value
```

**Note:** It's important to add `.env` to your `.gitignore` file to prevent accidentally committing sensitive information to your version control system.

**注意：** 将 `.env` 添加到您的 `.gitignore` 文件中很重要，以防止意外将敏感信息提交到版本控制系统。

```
# .gitignore
.env
```

### Using `cross-env`

### 使用 `cross-env`

`cross-env` is a popular npm package that allows you to set environment variables in a cross-platform way. This is useful because environment variable syntax can differ between operating systems (e.g., Windows vs. Linux/macOS).

`cross-env` 是一个流行的 npm 包，它允许您以跨平台的方式设置环境变量。这很有用，因为环境变量的语法在不同操作系统（例如 Windows 与 Linux/macOS）之间可能有所不同。

First, install `cross-env` as a dev dependency:

首先，将 `cross-env` 安装为开发依赖项：

```bash
npm install --save-dev cross-env
# or
yarn add --dev cross-env
```

Then, you can use it in your `package.json` scripts:

然后，您可以在 `package.json` 的脚本中使用它：

```json
{
  "scripts": {
    "start": "cross-env NODE_ENV=development node app.js",
    "prod": "cross-env NODE_ENV=production node app.js"
  }
}
```

Now you can run your scripts like:

现在您可以像这样运行您的脚本：

```bash
npm start
# or
yarn start
```

This will set the `NODE_ENV` variable to `development` before running `node app.js`.

这将为运行 `node app.js` 设置 `NODE_ENV` 变量为 `development`。

In [3]:
import os

os.environ["OPENAI_API_KEY"] = ""
os.environ["FIREWORKS_API_KEY"] = ""
os.environ["MONGO_URI"] = ""

FIREWORKS_API_KEY = os.environ.get("FIREWORKS_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
MONGO_URI = os.environ.get("MONGO_URI")

## 将数据摄入 MongoDB 向量数据库

MongoDB Atlas Search 现已支持向量搜索。这意味着您可以在 MongoDB 中存储和搜索向量嵌入。

本教程将指导您完成将数据摄入 MongoDB 向量数据库的过程。

**前提条件**

*   MongoDB Atlas 账户
*   已部署的 Atlas 集群
*   已安装 Python 和 pip
*   已安装 MongoDB Atlas Vector Search 客户端库：

```bash
pip install pymongo dnspython 'mongodb-atlas-search>=0.3.0'
```

**步骤**

1.  **准备数据**

    在本教程中，我们将使用一个包含文本描述的简单数据集。您可以使用自己的数据集，但请确保它包含您想要嵌入的文本字段。

    ```python
    data = [
        {"description": "A red car driving down a street."},
        {"description": "A blue car parked on a hill."},
        {"description": "A green car parked in a garage."},
        {"description": "A yellow car driving on a highway."},
        {"description": "A black car parked in a driveway."},
    ]
    ```

2.  **生成向量嵌入**

    您需要使用机器学习模型来生成文本描述的向量嵌入。在本教程中，我们将使用 Sentence Transformers 库。

    ```python
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer('all-MiniLM-L6-v2')

    for item in data:
        item['vector'] = model.encode(item['description']).tolist()
    ```

3.  **连接到 MongoDB Atlas**

    使用 `pymongo` 库连接到您的 MongoDB Atlas 集群。请将 `<your_cluster_url>` 替换为您的集群连接字符串。

    ```python
    from pymongo import MongoClient

    client = MongoClient("<your_cluster_url>")
    db = client.get_database("vector_db") # Replace with your database name
    collection = db.get_collection("documents") # Replace with your collection name
    ```

4.  **创建向量搜索索引**

    在您的集合上创建向量搜索索引。这将允许您对向量字段执行相似性搜索。

    ```python
    from mongodb_atlas_search.indexes import VectorSearchIndex

    index_name = "vector_index"
    vector_dimension = 384 # Dimension of the embeddings generated by 'all-MiniLM-L6-v2'
    index_definition = VectorSearchIndex(
        vector_dimension=vector_dimension,
        type="knn",
        paths=["vector"]
    ).definition()

    collection.create_search_index(index_name, index_definition)
    ```

5.  **将数据摄入 MongoDB**

    将包含向量嵌入的数据插入到您的 MongoDB 集合中。

    ```python
    collection.insert_many(data)
    ```

6.  **执行向量搜索**

    现在您可以对向量数据库执行相似性搜索了。首先，为您的查询生成向量嵌入。

    ```python
    query_text = "A fast car"
    query_vector = model.encode(query_text).tolist()
    ```

    然后，使用 `$vectorSearch` 聚合阶段执行搜索。

    ```python
    results = collection.aggregate([
        {
            "$vectorSearch": {
                "queryVector": query_vector,
                "path": "vector",
                "numCandidates": 5, # Number of nearest neighbors to return
                "index": index_name
            }
        }
    ])

    for result in results:
        print(result)
    ```

**示例输出**

```
{'_id': ObjectId('...'), 'description': 'A red car driving down a street.', 'vector': [...]}
{'_id': ObjectId('...'), 'description': 'A yellow car driving on a highway.', 'vector': [...]}
```

本教程介绍了将数据摄入 MongoDB 向量数据库的基础知识。您可以利用此功能构建强大的语义搜索应用程序。

In [2]:
import pandas as pd
from datasets import load_dataset

data = load_dataset("MongoDB/subset_arxiv_papers_with_emebeddings")
dataset_df = pd.DataFrame(data["train"])

/Users/richmondalake/miniconda3/envs/langchain_workarea/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Repo card metadata block was not found. Setting CardData to empty.
Generating train split: 50000 examples [00:01, 38699.64 examples/s]


In [4]:
print(len(dataset_df))
dataset_df.head()

50000


,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed,embedding
0,704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,...","[0.0594153292, -0.0440569334, -0.0487333685, -..."
1,704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]","[0.0247399714, -0.065658465, 0.0201423876, -0...."
2,704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is descri...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]","[0.0491479263, 0.0728017688, 0.0604138002, 0.0..."
3,704.0004,David Callan,David Callan,A determinant of Stirling cycle numbers counts...,11 pages,None,None,None,math.CO,None,We show that a determinant of Stirling cycle...,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2007-05-23,"[[Callan, David, ]]","[0.0389556214, -0.0410280302, 0.0410280302, -0..."
4,704.0005,Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,None,"Illinois J. Math. 52 (2008) no.2, 681-689",None,None,math.CA math.FA,None,In this paper we show how to compute the $\L...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2013-10-15,"[[Abu-Shammala, Wael, ], [Torchinsky, Alberto, ]]","[0.118412666, -0.0127423415, 0.1185125113, 0.0..."


In [5]:
from pymongo import MongoClient

# Initialize MongoDB python client
client = MongoClient(MONGO_URI, appname="devrel.content.ai_agent_firechain.python")

DB_NAME = "agent_demo"
COLLECTION_NAME = "knowledge"
ATLAS_VECTOR_SEARCH_INDEX_NAME = "vector_index"
collection = client[DB_NAME][COLLECTION_NAME]

In [6]:
# Delete any existing records in the collection
collection.delete_many({})

# Data Ingestion
records = dataset_df.to_dict("records")
collection.insert_many(records)

print("Data ingestion into MongoDB completed")

Data ingestion into MongoDB completed


## 创建向量搜索索引定义

```
{
  "fields": [
    {
      "type": "vector",
      "path": "embedding",
      "numDimensions": 256,
      "similarity": "cosine"
    }
  ]
}

## 创建 LangChain Retriever (MongoDB)

This document explains how to create a LangChain retriever that uses MongoDB as its data source.

本文档将介绍如何创建一个使用 MongoDB 作为数据源的 LangChain Retriever。

**Prerequisites**

**先决条件**

*   **MongoDB Atlas Account:** You need a MongoDB Atlas account and a cluster set up.
*   **MongoDB Atlas 账户：** 您需要一个 MongoDB Atlas 账户并已设置好集群。
*   **Python:** Ensure you have Python installed.
*   **Python：** 请确保您已安装 Python。
*   **LangChain:** Install the LangChain library:
    ```bash
    pip install langchain
    ```
*   **LangChain：** 安装 LangChain 库：
    ```bash
    pip install langchain
    ```
*   **pymongo:** Install the PyMongo driver for MongoDB:
    ```bash
    pip install pymongo
    ```
*   **pymongo：** 安装 PyMongo 驱动以连接 MongoDB：
    ```bash
    pip install pymongo
    ```
*   **Sentence Transformers:** Install Sentence Transformers for generating embeddings:
    ```bash
    pip install sentence-transformers
    ```
*   **Sentence Transformers：** 安装 Sentence Transformers 以生成嵌入向量：
    ```bash
    pip install sentence-transformers
    ```

**Steps**

**步骤**

1.  **Set up your MongoDB Atlas Cluster:**
    *   Create a free tier cluster on MongoDB Atlas if you don't have one.
    *   Create a database and a collection within your cluster.
    *   Add some sample data to your collection. Each document should ideally have a field for the text content and potentially other metadata.

    1.  **设置您的 MongoDB Atlas 集群：**
        *   如果您还没有 MongoDB Atlas 集群，请创建一个免费层级的集群。
        *   在您的集群中创建一个数据库和一个集合。
        *   向您的集合中添加一些示例数据。每个文档最好有一个用于文本内容的字段以及其他可能的元数据。

    2.  **Get your MongoDB Connection String:**
        *   Navigate to your MongoDB Atlas cluster.
        *   Click the "Connect" button.
        *   Choose "Connect your application".
        *   Select "Python" as the driver and the appropriate version.
        *   Copy the connection string. It will look something like this:
            ```
            mongodb+srv://<username>:<password>@<cluster-url>/<database-name>?retryWrites=true&w=majority
            ```
        *   Replace `<username>`, `<password>`, `<cluster-url>`, and `<database-name>` with your actual credentials and cluster details.

    2.  **获取您的 MongoDB 连接字符串：**
        *   导航到您的 MongoDB Atlas 集群。
        *   点击“Connect”按钮。
        *   选择“Connect your application”。
        *   选择“Python”作为驱动程序和相应的版本。
        *   复制连接字符串。它看起来会像这样：
            ```
            mongodb+srv://<username>:<password>@<cluster-url>/<database-name>?retryWrites=true&w=majority
            ```
        *   将 `<username>`、`<password>`、`<cluster-url>` 和 `<database-name>` 替换为您实际的凭据和集群详细信息。

    3.  **Create a Python Script:**
        *   Create a new Python file (e.g., `mongo_retriever.py`).
        *   Import the necessary libraries.

    3.  **创建 Python 脚本：**
        *   创建一个新的 Python 文件（例如 `mongo_retriever.py`）。
        *   导入必要的库。

    4.  **Initialize MongoDB Client and Collection:**
        *   Use `pymongo` to connect to your MongoDB Atlas cluster.
        *   Specify the database and collection you want to use.

    4.  **初始化 MongoDB 客户端和集合：**
        *   使用 `pymongo` 连接到您的 MongoDB Atlas 集群。
        *   指定您要使用的数据库和集合。

    5.  **Create the LangChain Retriever:**
        *   LangChain doesn't have a built-in MongoDB retriever. You'll need to create a custom retriever by subclassing `BaseRetriever`.
        *   Alternatively, you can use a simpler approach by creating a function that queries MongoDB and returns documents, then wrap it using `create_retriever_tool`.

    5.  **创建 LangChain Retriever：**
        *   LangChain 没有内置的 MongoDB Retriever。您需要通过继承 `BaseRetriever` 来创建一个自定义的 Retriever。
        *   或者，您可以使用一种更简单的方法，创建一个查询 MongoDB 并返回文档的函数，然后使用 `create_retriever_tool` 将其包装起来。

**Example Implementation (using `create_retriever_tool`)**

**示例实现（使用 `create_retriever_tool`）**

```python
import os
from pymongo import MongoClient
from langchain.tools.retriever import create_retriever_tool
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyMongoLoader
from langchain.schema.document import Document

# --- Configuration ---
MONGO_URI = os.environ.get("MONGO_URI", "mongodb+srv://<username>:<password>@<cluster-url>/<database-name>?retryWrites=true&w=majority")
DATABASE_NAME = "your_database_name"  # Replace with your database name
COLLECTION_NAME = "your_collection_name" # Replace with your collection name
TEXT_FIELD = "content" # The field in your MongoDB documents containing the text
METADATA_FIELDS = ["source", "page"] # Optional: Fields to include as metadata

# --- Initialize MongoDB Client ---
try:
    client = MongoClient(MONGO_URI)
    db = client[DATABASE_NAME]
    collection = db[COLLECTION_NAME]
    print("Successfully connected to MongoDB!")
except Exception as e:
    print(f"Error connecting to MongoDB: {e}")
    exit()

# --- Load Documents from MongoDB ---
# PyMongoLoader is a convenient way to load data from MongoDB
# It expects documents to be dictionaries. We convert them to LangChain Documents.
def load_docs_from_mongo():
    documents = []
    try:
        # Fetch all documents from the collection
        for doc in collection.find({}):
            # Extract text content from the specified TEXT_FIELD
            text_content = doc.get(TEXT_FIELD, "")
            if not text_content:
                continue # Skip documents without text content

            # Prepare metadata, including optional METADATA_FIELDS
            metadata = {}
            for field in METADATA_FIELDS:
                if field in doc:
                    metadata[field] = doc[field]
            # Add MongoDB's _id as a metadata field for reference
            metadata["_id"] = str(doc.get('_id'))

            documents.append(Document(page_content=text_content, metadata=metadata))
        print(f"Loaded {len(documents)} documents from MongoDB.")
        return documents
    except Exception as e:
        print(f"Error loading documents from MongoDB: {e}")
        return []

# --- Create Embeddings ---
# Using SentenceTransformerEmbeddings for generating vector embeddings
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# --- Create a Vector Store (e.g., FAISS) ---
# This step is optional if you want to perform similarity search directly on MongoDB
# However, for efficient retrieval with LangChain, a vector store is recommended.
# If your MongoDB documents already contain vector embeddings, you can adapt this.

# Load documents
mongo_documents = load_docs_from_mongo()

if not mongo_documents:
    print("No documents found or error loading documents. Exiting.")
    exit()

# Create FAISS index from loaded documents and embeddings
try:
    vector_store = FAISS.from_documents(mongo_documents, embeddings)
    print("FAISS vector store created successfully.")
except Exception as e:
    print(f"Error creating FAISS vector store: {e}")
    exit()

# --- Create the Retriever ---
# The retriever searches the vector store for relevant documents
retriever = vector_store.as_retriever()

# --- Create a Retriever Tool ---
# This tool can be used by an LLM agent
retriever_tool = create_retriever_tool(
    retriever,
    "mongo_document_retriever",
    "Searches and returns documents from MongoDB based on the query."
)

# --- Example Usage ---
if __name__ == "__main__":
    query = "What is the capital of France?" # Replace with your query
    print(f"\nSearching for: {query}")

    try:
        # Use the retriever tool to get relevant documents
        results = retriever_tool.func({"query": query}) # The tool expects a dict with a 'query' key

        if results:
            print("\nRetrieved Documents:")
            for i, doc in enumerate(results):
                print(f"--- Document {i+1} ---")
                print(f"Content: {doc.page_content[:200]}...") # Print first 200 chars
                print(f"Metadata: {doc.metadata}")
        else:
            print("No relevant documents found.")

    except Exception as e:
        print(f"An error occurred during retrieval: {e}")

    # Close the MongoDB connection
    client.close()
    print("\nMongoDB connection closed.")
```

**Explanation:**

1.  **Configuration:** Sets up MongoDB connection details, database/collection names, and the field containing text.
2.  **MongoDB Client Initialization:** Connects to your MongoDB Atlas cluster using `pymongo`.
3.  **`load_docs_from_mongo()` Function:**
    *   Queries the specified MongoDB collection.
    *   For each document, it extracts the text from the `TEXT_FIELD`.
    *   It constructs `Document` objects, including metadata from `METADATA_FIELDS` and the MongoDB `_id`.
4.  **Embeddings:** Initializes `SentenceTransformerEmbeddings` to convert text into numerical vector representations.
5.  **Vector Store (FAISS):**
    *   Loads the documents retrieved from MongoDB.
    *   Creates a FAISS index from these documents and their embeddings. FAISS is an efficient library for similarity search.
    *   **Note:** If your MongoDB documents already store vector embeddings (e.g., using MongoDB Atlas Vector Search), you would adapt this step to load those embeddings directly.
6.  **Retriever Creation:** Converts the FAISS vector store into a LangChain `retriever` object, which can efficiently search for similar documents based on a query's embedding.
7.  **Retriever Tool:** Uses `create_retriever_tool` to wrap the retriever, making it usable within LangChain's agent framework.
8.  **Example Usage (`if __name__ == "__main__":`)**
    *   Defines a sample query.
    *   Calls the `retriever_tool` with the query.
    *   Prints the content and metadata of the retrieved documents.
    *   Closes the MongoDB connection.

**Custom Retriever (Subclassing `BaseRetriever`)**

If you need more control or want to integrate directly with MongoDB's search capabilities (like Atlas Vector Search), you can create a custom retriever by subclassing `langchain.schema.BaseRetriever`.

**自定义 Retriever（继承 `BaseRetriever`）**

如果您需要更多控制权，或者想直接集成 MongoDB 的搜索功能（如 Atlas Vector Search），您可以继承 `langchain.schema.BaseRetriever` 来创建一个自定义的 Retriever。

```python
from typing import List, Dict, Any
from langchain.schema import BaseRetriever, Document
from langchain.embeddings import SentenceTransformerEmbeddings
from pymongo import MongoClient
import os

# --- Configuration ---
MONGO_URI = os.environ.get("MONGO_URI", "mongodb+srv://<username>:<password>@<cluster-url>/<database-name>?retryWrites=true&w=majority")
DATABASE_NAME = "your_database_name"
COLLECTION_NAME = "your_collection_name"
TEXT_FIELD = "content"
METADATA_FIELDS = ["source", "page"]

class MongoVectorRetriever(BaseRetriever):
    """Custom retriever that queries MongoDB using vector similarity."""

    collection: Any
    embeddings: Any
    text_key: str = TEXT_FIELD
    metadata_keys: List[str] = METADATA_FIELDS
    k: int = 4 # Number of documents to retrieve

    def _get_relevant_documents(self, query: str) -> List[Document]:
        """Retrieve documents from MongoDB based on query similarity."""
        try:
            # Generate embedding for the query
            query_embedding = self.embeddings.embed_query(query)

            # --- MongoDB Atlas Vector Search Integration ---
            # This assumes your MongoDB collection has a vector index configured
            # and a field (e.g., 'content_vector') storing the embeddings.
            # Adjust the pipeline according to your MongoDB setup.
            pipeline = [
                {
                    "$vectorSearch": {
                        "queryVector": query_embedding,
                        "path": "content_vector", # Replace with the actual field name storing embeddings
                        "numCandidates": self.k * 2, # Fetch more candidates for better results
                        "limit": self.k,
                        "index": "YourVectorSearchIndexName", # Replace with your vector index name
                    }
                },
                {
                    "$project": {
                        "_id": 0, # Exclude _id by default
                        "content": f"${self.text_key}", # Project the text field
                        "score": { "$meta": "vectorSearchScore" }, # Get the similarity score
                        **{key: f"${key}" for key in self.metadata_keys if key in self.collection.find_one() or True} # Project metadata fields
                    }
                }
            ]

            results = list(self.collection.aggregate(pipeline))

            # Convert MongoDB results to LangChain Documents
            docs = []
            for res in results:
                metadata = {key: res.get(key) for key in self.metadata_keys if key in res}
                # Add the similarity score to metadata if available
                if "score" in res:
                    metadata["score"] = res["score"]
                # Add MongoDB's _id if it was projected or available
                if '_id' in res:
                     metadata['_id'] = str(res['_id'])

                docs.append(Document(page_content=res.get("content", ""), metadata=metadata))

            return docs

        except Exception as e:
            print(f"Error during MongoDB vector search: {e}")
            return []

# --- Example Usage for Custom Retriever ---
if __name__ == "__main__":
    # Initialize MongoDB Client
    try:
        client = MongoClient(MONGO_URI)
        db = client[DATABASE_NAME]
        collection = db[COLLECTION_NAME]
        print("Successfully connected to MongoDB!")
    except Exception as e:
        print(f"Error connecting to MongoDB: {e}")
        exit()

    # Initialize Embeddings
    embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

    # Create the custom retriever instance
    # Ensure your MongoDB collection has a vector index and the 'content_vector' field
    custom_retriever = MongoVectorRetriever(
        collection=collection,
        embeddings=embeddings,
        text_key=TEXT_FIELD,
        metadata_keys=METADATA_FIELDS,
        k=3 # Retrieve top 3 results
    )

    query = "What are the benefits of AI?" # Replace with your query
    print(f"\nSearching for: {query}")

    try:
        # Use the custom retriever directly
        retrieved_docs = custom_retriever.get_relevant_documents(query)

        if retrieved_docs:
            print("\nRetrieved Documents (Custom Retriever):")
            for i, doc in enumerate(retrieved_docs):
                print(f"--- Document {i+1} ---")
                print(f"Content: {doc.page_content[:200]}...")
                print(f"Metadata: {doc.metadata}")
        else:
            print("No relevant documents found.")

    except Exception as e:
        print(f"An error occurred during retrieval: {e}")

    # Close the MongoDB connection
    client.close()
    print("\nMongoDB connection closed.")
```

**Explanation for Custom Retriever:**

1.  **`MongoVectorRetriever` Class:**
    *   Inherits from `BaseRetriever`.
    *   Requires `collection`, `embeddings`, `text_key`, `metadata_keys`, and `k` (number of results) as parameters.
2.  **`_get_relevant_documents(self, query: str)` Method:**
    *   This is the core method that LangChain calls for retrieval.
    *   It first generates an embedding for the input `query`.
    *   **MongoDB Atlas Vector Search:** It constructs an aggregation pipeline to perform a `$vectorSearch` directly on MongoDB.
        *   `queryVector`: The embedding of the user's query.
        *   `path`: The field in your MongoDB documents that stores the vector embeddings (e.g., `content_vector`). **You must replace `"content_vector"` with the actual field name in your collection.**
        *   `numCandidates`: Specifies how many documents to consider for the search.
        *   `limit`: The number of top results to return.
        *   `index`: The name of the vector search index you created in MongoDB Atlas. **You must replace `"YourVectorSearchIndexName"` with your actual index name.**
    *   The results from MongoDB are then converted into LangChain `Document` objects, including the text content and metadata.
3.  **Example Usage:** Demonstrates how to instantiate and use the `MongoVectorRetriever`.

**Important Considerations:**

*   **Vector Indexing in MongoDB:** For the custom retriever using `$vectorSearch`, you **must** have a vector search index configured in your MongoDB Atlas cluster. The `path` and `index` parameters in the `$vectorSearch` stage must match your index configuration.
*   **Embedding Storage:** Decide where your document embeddings are stored.
    *   If you're using FAISS (or another external vector store) as in the first example, you create and manage embeddings separately.
    *   If you're using MongoDB Atlas Vector Search, your embeddings are stored directly within MongoDB documents. You'll need a process to generate and store these embeddings when data is added or updated.
*   **Error Handling:** The provided examples include basic error handling. In a production environment, you'll want more robust error management.
*   **Security:** Store your `MONGO_URI` securely (e.g., using environment variables or a secrets management system) and avoid hardcoding credentials.
*   **Scalability:** For very large datasets, consider optimizing your MongoDB queries and indexing strategies. MongoDB Atlas Vector Search is designed for performance at scale.

**重要注意事项：**

*   **MongoDB 中的向量索引：** 对于使用 `$vectorSearch` 的自定义 Retriever，您**必须**在 MongoDB Atlas 集群中配置向量搜索索引。`$vectorSearch` 阶段中的 `path` 和 `index` 参数必须与您的索引配置匹配。
*   **嵌入存储：** 决定在哪里存储您的文档嵌入。
    *   如果您像第一个示例那样使用 FAISS（或其他外部向量存储），您将单独创建和管理嵌入。
    *   如果您使用 MongoDB Atlas Vector Search，您的嵌入将直接存储在 MongoDB 文档中。添加或更新数据时，您需要一个生成和存储这些嵌入的过程。
*   **错误处理：** 提供的示例包含基本的错误处理。在生产环境中，您需要更健壮的错误管理。
*   **安全性：** 安全地存储您的 `MONGO_URI`（例如，使用环境变量或密钥管理系统），并避免硬编码凭据。
*   **可扩展性：** 对于非常大的数据集，请考虑优化您的 MongoDB 查询和索引策略。MongoDB Atlas Vector Search 专为大规模性能而设计。

In [7]:
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=256)

# Vector Store Creation
vector_store = MongoDBAtlasVectorSearch.from_connection_string(
    connection_string=MONGO_URI,
    namespace=DB_NAME + "." + COLLECTION_NAME,
    embedding=embedding_model,
    index_name=ATLAS_VECTOR_SEARCH_INDEX_NAME,
    text_key="abstract",
)

retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

### 可选：使用 LLMLingua 创建具有压缩功能的检索器

In [ ]:
!pip install langchain_community llmlingua

In [9]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import LLMLinguaCompressor

In [12]:
compressor = LLMLinguaCompressor(model_name="openai-community/gpt2", device_map="cpu")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

/Users/richmondalake/miniconda3/envs/langchain_workarea/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## 使用 Fireworks AI 配置 LLM

This guide will walk you through the process of configuring a Large Language Model (LLM) using Fireworks AI.

本指南将引导您完成使用 Fireworks AI 配置大型语言模型 (LLM) 的过程。

### Prerequisites

在开始之前，请确保您已满足以下先决条件：

*   **An account with Fireworks AI:** If you don't have one, sign up at [https://fireworks.ai/](https://fireworks.ai/).
    *   **拥有 Fireworks AI 账户：** 如果您还没有账户，请在 [https://fireworks.ai/](https://fireworks.ai/) 注册。
*   **Your Fireworks AI API Key:** You can find this in your Fireworks AI account settings.
    *   **您的 Fireworks AI API 密钥：** 您可以在您的 Fireworks AI 账户设置中找到此密钥。
*   **Python installed:** Ensure you have Python 3.7 or later installed on your system.
    *   **已安装 Python：** 确保您的系统已安装 Python 3.7 或更高版本。
*   **`fireworks-ai` Python package installed:**
    ```bash
    pip install fireworks-ai
    ```
    *   **已安装 `fireworks-ai` Python 包：**
        ```bash
        pip install fireworks-ai
        ```

### Setting up your API Key

The `fireworks-ai` Python package automatically looks for your API key in the `FIREWORKS_API_KEY` environment variable.

`fireworks-ai` Python 包会自动在 `FIREWORKS_API_KEY` 环境变量中查找您的 API 密钥。

**Option 1: Set the environment variable directly**

You can set the environment variable in your terminal session:

**选项 1：直接设置环境变量**

您可以在终端会话中设置环境变量：

```bash
export FIREWORKS_API_KEY="your-api-key"
```

Replace `"your-api-key"` with your actual Fireworks AI API key.

将 `"your-api-key"` 替换为您的实际 Fireworks AI API 密钥。

**Option 2: Set the environment variable in your `.bashrc` or `.zshrc`**

To make the API key persistent, you can add it to your shell's configuration file (e.g., `.bashrc`, `.zshrc`):

**选项 2：在您的 `.bashrc` 或 `.zshrc` 中设置环境变量**

为了使 API 密钥持久化，您可以将其添加到您的 shell 配置文件中（例如 `.bashrc`、`.zshrc`）：

```bash
echo 'export FIREWORKS_API_KEY="your-api-key"' >> ~/.bashrc
source ~/.bashrc
```

Again, replace `"your-api-key"` with your actual API key.

同样，将 `"your-api-key"` 替换为您的实际 API 密钥。

### Using the Fireworks AI Client

Once your API key is set up, you can start using the Fireworks AI client to interact with LLMs.

设置好 API 密钥后，您就可以开始使用 Fireworks AI 客户端与 LLM 进行交互了。

Here's a basic example of how to generate text using a Fireworks AI model:

以下是一个使用 Fireworks AI 模型生成文本的基本示例：

```python
from fireworks.client import ChatCompletion

# Replace with the desired Fireworks AI model
# For example: "accounts/fireworks/models/llama-v2-70b-chat"
model_name = "accounts/fireworks/models/llama-v2-70b-chat"

try:
    response = ChatCompletion.create(
        model=model_name,
        messages=[
            {"role": "user", "content": "What is the capital of France?"}
        ]
    )
    print(response.choices[0].message.content)

except Exception as e:
    print(f"An error occurred: {e}")

```

**Explanation:**

*   **`from fireworks.client import ChatCompletion`**: This line imports the necessary `ChatCompletion` class from the `fireworks-ai` library.
    *   **`from fireworks.client import ChatCompletion`**: 此行从 `fireworks-ai` 库导入必要的 `ChatCompletion` 类。
*   **`model_name = "accounts/fireworks/models/llama-v2-70b-chat"`**: This variable holds the identifier for the specific Fireworks AI model you want to use. You can find a list of available models on the Fireworks AI platform.
    *   **`model_name = "accounts/fireworks/models/llama-v2-70b-chat"`**: 此变量包含您要使用的特定 Fireworks AI 模型的标识符。您可以在 Fireworks AI 平台上找到可用模型的列表。
*   **`ChatCompletion.create(...)`**: This is the core function call to generate a response from the LLM.
    *   **`ChatCompletion.create(...)`**: 这是从 LLM 生成响应的核心函数调用。
*   **`model=model_name`**: Specifies which model to use for the generation.
    *   **`model=model_name`**: 指定要用于生成的模型。
*   **`messages=[{"role": "user", "content": "What is the capital of France?"}]`**: This is a list of messages representing the conversation history. In this case, it's a single user message.
    *   **`messages=[{"role": "user", "content": "What is the capital of France?"}]`**: 这是一个代表对话历史的消息列表。在这种情况下，它是一个用户消息。
*   **`response.choices[0].message.content`**: This accesses the generated text content from the model's response.
    *   **`response.choices[0].message.content`**: 这可以访问模型响应中生成的文本内容。
*   **`try...except` block**: This is used for basic error handling.
    *   **`try...except` 块**: 这用于基本错误处理。

### Available Models

Fireworks AI offers a variety of LLMs. You can find the most up-to-date list of available models and their identifiers on the Fireworks AI website or by using their API. Some popular models include:

Fireworks AI 提供多种 LLM。您可以在 Fireworks AI 网站上或通过其 API 找到可用模型及其标识符的最新列表。一些流行的模型包括：

*   `accounts/fireworks/models/llama-v2-70b-chat`
*   `accounts/fireworks/models/mixtral-8x7b-instruct-v0.1`
*   `accounts/fireworks/models/phi-2`

Remember to replace `"accounts/fireworks/models/llama-v2-70b-chat"` with the identifier of the model you wish to use.

请记住将 `"accounts/fireworks/models/llama-v2-70b-chat"` 替换为您希望使用的模型的标识符。

### Further Customization

The `ChatCompletion.create` method accepts various parameters to customize the LLM's behavior, such as:

`ChatCompletion.create` 方法接受各种参数来定制 LLM 的行为，例如：

*   **`temperature`**: Controls the randomness of the output. Higher values (e.g., 0.8) make the output more random, while lower values (e.g., 0.2) make it more deterministic.
    *   **`temperature`**: 控制输出的随机性。较高的值（例如 0.8）使输出更随机，而较低的值（例如 0.2）使其更确定。
*   **`max_tokens`**: The maximum number of tokens to generate in the response.
    *   **`max_tokens`**: 在响应中生成的最大令牌数。
*   **`top_p`**: Nucleus sampling parameter.
    *   **`top_p`**: 核心采样参数。

Here's an example with customization:

以下是一个包含自定义的示例：

```python
from fireworks.client import ChatCompletion

model_name = "accounts/fireworks/models/llama-v2-70b-chat"

try:
    response = ChatCompletion.create(
        model=model_name,
        messages=[
            {"role": "user", "content": "Tell me a short story about a brave knight."}
        ],
        temperature=0.7,
        max_tokens=150,
        top_p=0.9
    )
    print(response.choices[0].message.content)

except Exception as e:
    print(f"An error occurred: {e}")
```

This setup allows you to easily integrate Fireworks AI's powerful LLMs into your Python applications.

此设置使您可以轻松地将 Fireworks AI 的强大 LLM 集成到您的 Python 应用程序中。

In [61]:
from langchain_fireworks import ChatFireworks

llm = ChatFireworks(model="accounts/fireworks/models/firefunction-v1", max_tokens=256)

## Agent 工具创建

This guide will walk you through the process of creating custom tools for your agents.

本指南将引导您完成为 Agent 创建自定义工具的过程。

Tools are functions that an agent can use to interact with the outside world. They can be used to fetch data, perform actions, or access external APIs.

工具是 Agent 可以用来与外部世界交互的函数。它们可用于获取数据、执行操作或访问外部 API。

### 1. Defining a Tool

To define a tool, you need to create a Python function and decorate it with `@tool`.

要定义一个工具，您需要创建一个 Python 函数并使用 `@tool` 进行装饰。

```python
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.agent import AgentExecutor, AgentState

# Define a simple tool
@tool
def get_current_weather(location: str) -> str:
    """
    Returns the current weather for a given location.
    """
    # In a real application, you would fetch this from a weather API
    weather_info = {
        "tokyo": "It's sunny and 25 degrees Celsius.",
        "new york": "It's rainy and 15 degrees Celsius.",
        "london": "It's cloudy and 18 degrees Celsius.",
    }
    return weather_info.get(location.lower(), "Weather information not available for that location.")

# Example of how to use the tool within an agent
# (This is a simplified example, a full agent setup is more complex)
# agent_executor = AgentExecutor(tools=[get_current_weather], ...)
```

### 2. Tool Schema

The tool's schema is automatically inferred from the function's signature and docstring. The docstring should clearly describe what the tool does and what arguments it expects.

工具的模式会从函数的签名和文档字符串中自动推断出来。文档字符串应清楚地描述工具的功能以及它期望的参数。

The `tool` decorator uses Pydantic to define the schema, allowing for type hints and validation.

`tool` 装饰器使用 Pydantic 来定义模式，支持类型提示和验证。

```python
from typing import Literal

# Example with more complex schema
@tool
def search_api(query: str, sort_by: Literal["relevance", "date"] = "relevance") -> str:
    """
    Searches an API for information based on a query.

    Args:
        query: The search query.
        sort_by: How to sort the search results. Defaults to "relevance".
    """
    return f"Searching API for '{query}' sorted by {sort_by}..."

# The schema for search_api would include:
# - query: str
# - sort_by: Literal["relevance", "date"] (optional, defaults to "relevance")
```

### 3. Registering Tools

Tools need to be registered with the agent so that the agent knows they are available. This is typically done by passing a list of tools to the agent's constructor or a configuration object.

工具需要注册到 Agent，以便 Agent 知道它们是可用的。这通常通过将工具列表传递给 Agent 的构造函数或配置对象来完成。

```python
from langgraph.graph.agent import AgentExecutor
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver

# Assuming get_current_weather and search_api are defined as above

# Create an agent executor with the defined tools
# agent_executor = AgentExecutor(
#     tools=[get_current_weather, search_api],
#     # ... other agent configurations
# )
```

### 4. Tool Usage in Agent Logic

An agent can decide to use a tool based on the current state or input. This decision-making process is part of the agent's core logic. When a tool is selected, the agent will call the tool with the appropriate arguments.

Agent 可以根据当前状态或输入决定使用哪个工具。这个决策过程是 Agent 核心逻辑的一部分。当选择一个工具时，Agent 将使用适当的参数调用该工具。

```python
# Example of agent logic (conceptual)
# def agent_node(state: AgentState):
#     user_input = state["input"]
#     if "weather" in user_input:
#         location = extract_location(user_input) # Assume this function exists
#         return {"tool_calls": [get_current_weather.bind(location=location)]}
#     elif "search" in user_input:
#         query = extract_query(user_input) # Assume this function exists
#         return {"tool_calls": [search_api.bind(query=query)]}
#     else:
#         return {"messages": ["I don't know how to help with that."]}
```

### 5. Tool Output Handling

The output from a tool call is returned to the agent, which can then use this information to formulate a response or decide on the next action.

工具调用的输出会返回给 Agent，Agent 随后可以使用这些信息来制定响应或决定下一步操作。

```python
# Example of handling tool output (conceptual)
# def process_tool_output(state: AgentState):
#     tool_result = state["tool_result"]
#     # Process the tool_result (e.g., display weather, show search results)
#     return {"messages": [f"Tool returned: {tool_result}"]}
```

### Advanced Tool Creation

#### Custom Tool Input/Output Schemas

You can define more complex input and output schemas using Pydantic models.

您可以使用 Pydantic 模型定义更复杂的输入和输出模式。

```python
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    location: str = Field(description="The city and state, e.g. San Francisco, CA")

class WeatherOutput(BaseModel):
    temperature: float
    unit: Literal["celsius", "fahrenheit"]
    description: str

@tool(input_schema=WeatherInput, output_schema=WeatherOutput)
def get_weather_with_schema(location: str) -> WeatherOutput:
    """
    Gets the current weather for a given location.
    """
    # Mock API call
    if "tokyo" in location.lower():
        return WeatherOutput(temperature=25.0, unit="celsius", description="Sunny")
    elif "new york" in location.lower():
        return WeatherOutput(temperature=70.0, unit="fahrenheit", description="Rainy")
    else:
        raise ValueError("Location not found")

# The agent will now expect the tool to conform to WeatherInput and WeatherOutput schemas.
```

#### Async Tools

For tools that involve I/O operations (like network requests), you can define them as asynchronous functions.

对于涉及 I/O 操作（如网络请求）的工具，您可以将其定义为异步函数。

```python
import asyncio

@tool
async def async_get_data(url: str) -> str:
    """
    Fetches data from a given URL asynchronously.
    """
    # Simulate an async network request
    await asyncio.sleep(1)
    return f"Data from {url}"

# AgentExecutor will automatically handle calling async tools.
```

#### Tool Configuration

You can pass additional configuration to tools, such as descriptions or metadata, which can be used by the agent for better decision-making.

您可以向工具传递额外的配置，例如描述或元数据，Agent 可以使用这些信息来做出更好的决策。

```python
@tool(description="A tool to perform complex calculations.")
def calculate(expression: str) -> str:
    """
    Evaluates a mathematical expression.
    """
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

In [51]:
from langchain.agents import tool
from langchain.tools.retriever import create_retriever_tool
from langchain_community.document_loaders import ArxivLoader


# Custom Tool Definiton
@tool
def get_metadata_information_from_arxiv(word: str) -> list:
    """
    Fetches and returns metadata for a maximum of ten documents from arXiv matching the given query word.

    Args:
      word (str): The search query to find relevant documents on arXiv.

    Returns:
      list: Metadata about the documents matching the query.
    """
    docs = ArxivLoader(query=word, load_max_docs=10).load()
    # Extract just the metadata from each document
    metadata_list = [doc.metadata for doc in docs]
    return metadata_list


@tool
def get_information_from_arxiv(word: str) -> list:
    """
    Fetches and returns metadata for a single research paper from arXiv matching the given query word, which is the ID of the paper, for example: 704.0001.

    Args:
      word (str): The search query to find the relevant paper on arXiv using the ID.

    Returns:
      list: Data about the paper matching the query.
    """
    doc = ArxivLoader(query=word, load_max_docs=1).load()
    return doc


# If you created a retriever with compression capaitilies in the optional cell in an earlier cell, you can replace 'retriever' with 'compression_retriever'
# Otherwise you can also create a compression procedure as a tool for the agent as shown in the `compress_prompt_using_llmlingua` tool definition function
retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="knowledge_base",
    description="This serves as the base knowledge source of the agent and contains some records of research papers from Arxiv. This tool is used as the first step for exploration and reseach efforts.",
)

In [52]:
from langchain_community.document_compressors import LLMLinguaCompressor

compressor = LLMLinguaCompressor(model_name="openai-community/gpt2", device_map="cpu")


@tool
def compress_prompt_using_llmlingua(prompt: str, compression_rate: float = 0.5) -> str:
    """
    Compresses a long data or prompt using the LLMLinguaCompressor.

    Args:
        data (str): The data or prompt to be compressed.
        compression_rate (float): The rate at which to compress the data (default is 0.5).

    Returns:
        str: The compressed data or prompt.
    """
    compressed_data = compressor.compress_prompt(
        prompt,
        rate=compression_rate,
        force_tokens=["!", ".", "?", "\n"],
        drop_consecutive=True,
    )
    return compressed_data

In [53]:
tools = [
    retriever_tool,
    get_metadata_information_from_arxiv,
    get_information_from_arxiv,
    compress_prompt_using_llmlingua,
]

## Agent Prompt 创建

**Agent Prompt** 是一个用于指导 AI Agent 如何执行特定任务的指令。它通常包含以下几个部分：

*   **角色 (Role):** 定义 AI Agent 的身份和职责。例如，“你是一名经验丰富的软件工程师。”
*   **任务 (Task):** 明确 AI Agent 需要完成的具体工作。例如，“编写一个 Python 函数来计算斐波那契数列。”
*   **上下文 (Context):** 提供完成任务所需的背景信息。例如，“该函数将被用于一个需要高效计算的场景。”
*   **约束 (Constraints):** 设定 AI Agent 在执行任务时需要遵守的规则或限制。例如，“函数应接受一个整数作为输入，并返回一个整数。”
*   **输出格式 (Output Format):** 指定 AI Agent 输出结果的格式。例如，“输出应为 Python 代码块，并包含适当的注释。”

以下是一个 Agent Prompt 的示例：

```
你是一名经验丰富的软件工程师。

你的任务是编写一个 Python 函数来计算斐波那契数列。

该函数将被用于一个需要高效计算的场景。

函数应接受一个整数作为输入，并返回一个整数。

输出应为 Python 代码块，并包含适当的注释。
```

**创建有效的 Agent Prompt 的技巧：**

*   **清晰具体:** 确保指令清晰、具体，避免模糊的语言。
*   **提供足够信息:** 提供完成任务所需的全部信息，包括任何必要的背景知识或示例。
*   **设定明确的期望:** 明确你对 AI Agent 输出的期望，包括格式、风格和内容。
*   **迭代和优化:** 根据 AI Agent 的输出，不断迭代和优化你的 Prompt，以获得更好的结果。
*   **考虑边缘情况:** 思考可能出现的边缘情况或错误输入，并在 Prompt 中加以说明。

**Agent Prompt 的应用场景：**

Agent Prompt 可用于各种 AI Agent 任务，包括：

*   **代码生成:** 指导 AI Agent 编写特定功能的代码。
*   **文本创作:** 指导 AI Agent 撰写文章、故事或邮件。
*   **数据分析:** 指导 AI Agent 分析数据并提取见解。
*   **问题解答:** 指导 AI Agent 回答特定领域的问题。
*   **任务自动化:** 指导 AI Agent 自动执行重复性任务。

通过精心设计的 Agent Prompt，您可以有效地引导 AI Agent 完成各种复杂的任务，从而提高效率和生产力。

In [89]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

agent_purpose = """
You are a helpful research assistant equipped with various tools to assist with your tasks efficiently. 
You have access to conversational history stored in your inpout as chat_history.
You are cost-effective and utilize the compress_prompt_using_llmlingua tool whenever you determine that a prompt or conversational history is too long. 
Below are instructions on when and how to use each tool in your operations.

1. get_metadata_information_from_arxiv

Purpose: To fetch and return metadata for up to ten documents from arXiv that match a given query word.
When to Use: Use this tool when you need to gather metadata about multiple research papers related to a specific topic.
Example: If you are asked to provide an overview of recent papers on "machine learning," use this tool to fetch metadata for relevant documents.

2. get_information_from_arxiv

Purpose: To fetch and return metadata for a single research paper from arXiv using the paper's ID.
When to Use: Use this tool when you need detailed information about a specific research paper identified by its arXiv ID.
Example: If you are asked to retrieve detailed information about the paper with the ID "704.0001," use this tool.

3. retriever_tool

Purpose: To serve as your base knowledge, containing records of research papers from arXiv.
When to Use: Use this tool as the first step for exploration and research efforts when dealing with topics covered by the documents in the knowledge base.
Example: When beginning research on a new topic that is well-documented in the arXiv repository, use this tool to access the relevant papers.

4. compress_prompt_using_llmlingua

Purpose: To compress long prompts or conversational histories using the LLMLinguaCompressor.
When to Use: Use this tool whenever you determine that a prompt or conversational history is too long to be efficiently processed.
Example: If you receive a very lengthy query or conversation context that exceeds the typical token limits, compress it using this tool before proceeding with further processing.

"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", agent_purpose),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

## 使用 MongoDB 实现 Agent 记忆功能

In [92]:
from langchain.memory import ConversationBufferMemory
from langchain_mongodb.chat_message_histories import MongoDBChatMessageHistory


def get_session_history(session_id: str) -> MongoDBChatMessageHistory:
    return MongoDBChatMessageHistory(
        MONGO_URI, session_id, database_name=DB_NAME, collection_name="history"
    )


memory = ConversationBufferMemory(
    memory_key="chat_history", chat_memory=get_session_history("latest_agent_session")
)

## 代理创建

In [93]:
from langchain.agents import AgentExecutor, create_tool_calling_agent

agent = create_tool_calling_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    memory=memory,
)

## Agent 执行

An agent is a program that can perform actions on your behalf. It can access and interact with your data, and it can also execute code. Agents are a powerful tool for automating tasks and building intelligent applications.

Agent execution is the process by which an agent performs its tasks. This process typically involves the following steps:

1. **Receiving a request:** An agent receives a request from a user or another program. The request typically specifies the task that the agent should perform.
2. **Planning the task:** The agent analyzes the request and creates a plan for how to perform the task. This plan may involve breaking the task down into smaller subtasks.
3. **Executing the plan:** The agent executes the plan by performing the necessary actions. This may involve accessing data, executing code, or interacting with other programs.
4. **Returning the results:** Once the agent has completed the task, it returns the results to the user or program that made the request.

Here are some examples of how agents can be used:

* **Automating tasks:** Agents can be used to automate repetitive tasks, such as sending emails, scheduling appointments, or generating reports.
* **Building intelligent applications:** Agents can be used to build intelligent applications that can learn from data and make decisions. For example, an agent could be used to build a chatbot that can answer customer questions or a recommendation system that can suggest products to users.
* **Interacting with data:** Agents can be used to interact with data in a variety of ways, such as querying databases, analyzing spreadsheets, or visualizing data.

Agent execution is a powerful tool that can be used to automate tasks, build intelligent applications, and interact with data. By understanding how agents work, you can leverage their power to improve your productivity and build more innovative applications.

---

Agent 是一个可以代表您执行操作的程序。它可以访问和交互您的数据，还可以执行代码。Agent 是自动化任务和构建智能应用程序的强大工具。

Agent 执行是 Agent 执行其任务的过程。此过程通常涉及以下步骤：

1. **接收请求：** Agent 从用户或其他程序接收请求。请求通常指定 Agent 应执行的任务。
2. **规划任务：** Agent 分析请求并制定执行任务的计划。此计划可能涉及将任务分解为更小的子任务。
3. **执行计划：** Agent 通过执行必要的操作来执行计划。这可能涉及访问数据、执行代码或与其他程序交互。
4. **返回结果：** Agent 完成任务后，会将结果返回给提出请求的用户或程序。

以下是一些 Agent 用途的示例：

* **自动化任务：** Agent 可用于自动化重复性任务，例如发送电子邮件、安排约会或生成报告。
* **构建智能应用程序：** Agent 可用于构建可以从数据中学习并做出决策的智能应用程序。例如，Agent 可用于构建可以回答客户问题的聊天机器人，或可以向用户推荐产品的推荐系统。
* **与数据交互：** Agent 可用于以多种方式与数据交互，例如查询数据库、分析电子表格或可视化数据。

Agent 执行是一个强大的工具，可用于自动化任务、构建智能应用程序和与数据交互。通过了解 Agent 的工作原理，您可以利用它们的强大功能来提高工作效率并构建更具创新性的应用程序。

In [94]:
agent_executor.invoke(
    {
        "input": "Get me a list of research papers on the topic Prompt Compression in LLM Applications."
    }
)



> Entering new AgentExecutor chain...

Invoking: `get_metadata_information_from_arxiv` with `{'word': 'Prompt Compression in LLM Applications'}`


[{'Published': '2024-05-27', 'Title': 'SelfCP: Compressing Long Prompt to 1/12 Using the Frozen Large Language Model Itself', 'Authors': 'Jun Gao', 'Summary': 'Long prompt leads to huge hardware costs when using Large Language Models\n(LLMs). Unfortunately, many tasks, such as summarization, inevitably introduce\nlong task-inputs, and the wide application of in-context learning easily makes\nthe prompt length explode. Inspired by the language understanding ability of\nLLMs, this paper proposes SelfCP, which uses the LLM \\textbf{itself} to\n\\textbf{C}ompress long \\textbf{P}rompt into compact virtual tokens. SelfCP\napplies a general frozen LLM twice, first as an encoder to compress the prompt\nand then as a decoder to generate responses. Specifically, given a long prompt,\nwe place special tokens within the lengthy segment for compressio

{'input': 'Get me a list of research papers on the topic Prompt Compression in LLM Applications.',
 'chat_history': '',
 'output': 'Here are some research papers on the topic Prompt Compression in LLM Applications:\n\n1. "SelfCP: Compressing Long Prompt to 1/12 Using the Frozen Large Language Model Itself" by Jun Gao\n2. "Adapting LLMs for Efficient Context Processing through Soft Prompt Compression" by Cangqing Wang, Yutian Yang, Ruisi Li, Dan Sun, Ruicong Cai, Yuzhu Zhang, Chengqian Fu, Lillian Floyd\n3. "LLMLingua: Compressing Prompts for Accelerated Inference of Large Language Models" by Huiqiang Jiang, Qianhui Wu, Chin-Yew Lin, Yuqing Yang, Lili Qiu\n4. "Learning to Compress Prompt in Natural Language Formats" by Yu-Neng Chuang, Tianwei Xing, Chia-Yuan Chang, Zirui Liu, Xun Chen, Xia Hu\n5. "PROMPT-SAW: Leveraging Relation-Aware Graphs for Textual Prompt Compression"'}

In [95]:
agent_executor.invoke({"input": "What paper did we speak about from our chat history?"})



> Entering new AgentExecutor chain...

Invoking: `get_metadata_information_from_arxiv` with `{'word': 'chat history'}`
responded: I need to access the chat history to answer this question. 

[{'Published': '2023-10-20', 'Title': 'Towards Detecting Contextual Real-Time Toxicity for In-Game Chat', 'Authors': 'Zachary Yang, Nicolas Grenan-Godbout, Reihaneh Rabbany', 'Summary': "Real-time toxicity detection in online environments poses a significant\nchallenge, due to the increasing prevalence of social media and gaming\nplatforms. We introduce ToxBuster, a simple and scalable model that reliably\ndetects toxic content in real-time for a line of chat by including chat history\nand metadata. ToxBuster consistently outperforms conventional toxicity models\nacross popular multiplayer games, including Rainbow Six Siege, For Honor, and\nDOTA 2. We conduct an ablation study to assess the importance of each model\ncomponent and explore ToxBuster's transferability across the datasets.\nFurthermo

{'input': 'What paper did we speak about from our chat history?',
 'chat_history': 'Human: Get me a list of research papers on the topic Prompt Compression in LLM Applications.\nAI: Here are some research papers on the topic Prompt Compression in LLM Applications:\n\n1. "SelfCP: Compressing Long Prompt to 1/12 Using the Frozen Large Language Model Itself" by Jun Gao\n2. "Adapting LLMs for Efficient Context Processing through Soft Prompt Compression" by Cangqing Wang, Yutian Yang, Ruisi Li, Dan Sun, Ruicong Cai, Yuzhu Zhang, Chengqian Fu, Lillian Floyd\n3. "LLMLingua: Compressing Prompts for Accelerated Inference of Large Language Models" by Huiqiang Jiang, Qianhui Wu, Chin-Yew Lin, Yuqing Yang, Lili Qiu\n4. "Learning to Compress Prompt in Natural Language Formats" by Yu-Neng Chuang, Tianwei Xing, Chia-Yuan Chang, Zirui Liu, Xun Chen, Xia Hu\n5. "PROMPT-SAW: Leveraging Relation-Aware Graphs for Textual Prompt Compression"',
 'output': 'The paper we spoke about from our chat history is "